# Notebook 1: Koalescenten som Phase-type fordeling

I denne notebook ser der nærmere på Kingmans koalescent som en kontinuert phase-type fordeling implementeret via phasic. Jeg ser på kendte analytiske resultater numerisk og undersøger, hvordan centrale populationsgenetiske størrelser: træ højde, samlet grenlængde og site frequency spectrum (SFS) kan udtrykkes som reward-transformationer af samme underliggende Markov-proces.

In [ ]:
# Importer nødvendige pakker
from phasic import Graph, with_ipv  
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%config InlineBackend.figure_format = 'svg'

np.random.seed(42)
sns.set_palette('tab10')
plt.rcParams['figure.figsize'] = (8, 4)

## Del 1 – Koalescent-modellen

Koalescenten er en kontinuert Markov-kæde, der beskriver ancestrale linjer baglæns i tid. Med $n$ samples starter processen i tilstanden $(n, 0, \ldots, 0)$, hvor alle lineager er singletons. To lineager smelter sammen (koalescerer) med rate $\binom{k}{2} = \frac{k(k-1)}{2}$ når der er $k$ lineager til stede. Tilstandsvektoren holder styr på antallet af lineager med $1, 2, 3, \ldots$ efterkommere.

Koalescenten er phase-type fordelt: absorptionstidspunktet (TMRCA) er $\tau \sim \text{PH}(\alpha, \mathbf{T})$, hvor $\alpha = (1, 0, \ldots, 0)$ og $\mathbf{T}$ er sub-intensitetsmatricen.

### Spørgsmål 1

 - Kan jeg verificere, at koalescenten er korrekt implementeret i phasic ved at sammenligne numeriske og analytiske resultater for $\mathbb{E}[H]$ og $\text{Var}[H]$?

Hypotese: For et sample af størrelse $n$ er: 
$$
\mathbb{E}[H] = 2\left(1 - \frac{1}{n}\right), \quad \text{Var}[H] = 4\sum_{k=2}^{n} \frac{1}{k^2(k-1)^2/4}
$$

In [ ]:
# Koalescent callback-funktion
# Tilstandsvektoren: state[i] = antal lineager med (i+1) efterkommere
nr_samples = 4

@with_ipv([nr_samples] + [0] * (nr_samples - 1))
def coalescent(state):
    transitions = []
    for i in range(state.size):
        for j in range(i, state.size):
            same = int(i == j)
            if same and state[i] < 2:
                continue
            if not same and (state[i] < 1 or state[j] < 1):
                continue
            new = state.copy()
            new[i] -= 1
            new[j] -= 1
            new[i + j + 1] += 1
            transitions.append((new, state[i] * (state[j] - same) / (1 + same)))
    return transitions

graph = Graph(coalescent)
graph.plot()

In [ ]:
# Vis tilstandsrummet som DataFrame
labels = [f"{i+1}'ton" for i in range(nr_samples)]
pd.DataFrame(graph.states(), columns=labels)

In [ ]:
# Verificering: E[H] og Var[H] numerisk vs analytisk
print("=== Verificering for n=4 ===")
print(f"Numerisk  E[H] = {graph.expectation():.6f}")
print(f"Analytisk E[H] = 2*(1-1/n) = {2*(1-1/nr_samples):.6f}")
print(f"Numerisk  Var[H] = {graph.variance():.6f}")
print()

# Sammenlign for n = 2 til 8
results = []
for n in range(2, 9):
    @with_ipv([n] + [0] * (n - 1))
    def coal_n(state):
        transitions = []
        for i in range(state.size):
            for j in range(i, state.size):
                same = int(i == j)
                if same and state[i] < 2: continue
                if not same and (state[i] < 1 or state[j] < 1): continue
                new = state.copy()
                new[i] -= 1; new[j] -= 1; new[i+j+1] += 1
                transitions.append((new, state[i]*(state[j]-same)/(1+same)))
        return transitions
    g = Graph(coal_n)
    E_num = g.expectation()
    E_th  = 2 * (1 - 1/n)
    results.append({'n': n, 'E[H] numerisk': round(E_num,5), 'E[H] analytisk': round(E_th,5),
                    'Forskel': round(abs(E_num - E_th), 8)})

pd.DataFrame(results).set_index('n')

## Del 2 – Expected sojourn time

Den forventede sojourn time (opoldstid) i en tilstand svarer til den forventede tid processen tilbringer i den tilstand inden absorption. For koalescenten svarer dette til den forventede tid med præcis $k$ lineager til stede.

### Spørgsmål 2

- Er expected sojourn time i tilstanden med $k$ lineager lig $\frac{2}{k(k-1)}$ dvs. den reciprokke koalescensrate?

Hypotese: 

- Da koalescensraten med $k$ lineager er $\binom{k}{2} = k(k-1)/2$, er den forventede opholdstid $\frac{1}{k(k-1)/2} = \frac{2}{k(k-1)}$. Jeg forventer, at *graph.expected_sojourn_time()* bekræfter dette.

In [ ]:
graph = Graph(coalescent)
sojourn = graph.expected_sojourn_time()
states  = graph.states()

print(f"{'Tilstand':<30} {'k lin.':<8} {'Sojourn num.':<16} {'2/k(k-1) analytisk':<20}")
print("-" * 76)
for i, (s, t) in enumerate(zip(states, sojourn)):
    k = int(sum(s))
    th = 2 / (k * (k - 1)) if k >= 2 else 0.0
    state_str = str(dict(zip(labels, s)))
    print(f"{state_str:<30} {k:<8} {t:<16.5f} {th:.5f}")

In [ ]:
# Verificer: sum af sojourn times = E[H]
print(f"Sum af sojourn times = {sum(sojourn):.6f}")
print(f"graph.expectation()  = {graph.expectation():.6f}")
print(f"Er de ens? {np.isclose(sum(sojourn), graph.expectation())}")

## Del 3 – Reward-transformationer: SFS og træ længde

En reward-transformation lader mig beregne den forventede akkumulerede tid brugt i bestemte tilstande vægtet med en reward-vektor $\mathbf{r}$. Dette giver mig direkte adgang til populationsgenetiske størrelser som samlede grenlængde og site frequency spectrum (SFS).

Den samlede tid brugt i tilstande med $i$ linjer af en bestemt type er proportional med den forventede træ længde for den pågældende kategori.

### Spørgsmål 3

- Er E[samlede grenlængde] lig Wattersons sum $2\sum_{k=1}^{n-1} \frac{1}{k}$?**

Hypotese: 
 - Total branch length $L = \sum_{k=2}^{n} k \cdot T_k$, hvor $T_k \sim \text{Exp}(k(k-1)/2)$. Den forventede samlede grenlængde er $\mathbb{E}[L] = 2\sum_{k=1}^{n-1} \frac{1}{k}$ (Wattersons formel), som bruges i estimatoren $\hat{\theta}_W = S / a_n$.

In [ ]:
graph = Graph(coalescent)
reward_matrix = graph.states().T  # shape: (n_types, n_states)

# SFS: forventet branch length per lineage-type
sfs = [graph.expectation(r) for r in reward_matrix]
print("Site Frequency Spectrum (forventet branch length):")
for lab, val in zip(labels, sfs):
    print(f"  {lab}: {val:.6f}")

# Total branch length = vægtet sum
total_brlen_reward = reward_matrix.sum(axis=0)
E_L_num = graph.expectation(total_brlen_reward)
E_L_th  = 2 * sum(1/k for k in range(1, nr_samples))

print(f"\nE[L] numerisk  = {E_L_num:.6f}")
print(f"E[L] analytisk = 2*sum(1/k, k=1..{nr_samples-1}) = {E_L_th:.6f}")
print(f"Wattersons a_n = {E_L_th/2:.6f}")

In [ ]:
# Plot SFS
fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=labels, y=sfs, hue=labels, ax=ax, legend=False)
ax.set_xlabel("Lineage type")
ax.set_ylabel("Forventet branch length")
ax.set_title(f"Site Frequency Spectrum (n={nr_samples})")
plt.tight_layout()
plt.show()

## Del 4 – Kovariansstruktur i SFS

Med den multivariate phase-type ramme kan jeg beregne kovarianser og korrelationer mellem gren længde direkte uden simulering. Dette er en af de store fordele ved phase-type teorien frem for klassiske tilgange.

### Spørgsmål 4

- Er singleton og doubleton gren længde negativt korrelerede? Og er kovariansen mere negativ for branches der deler mere tid?

Hypotese: 

 - Da singleton og doubleton gren længde begge akkumuleres i de tidlige stadier af koalescenten, forventer jeg negativ kovarians tid brugt på et trækker fra de andre. Jeg forventer $\text{Cov}(B_1, B_2) < 0$.

In [ ]:
# Kovariansmatrix for alle SFS-komponenter
n_types = reward_matrix.shape[0]
cov_matrix = np.zeros((n_types, n_types))
for i in range(n_types):
    for j in range(n_types):
        cov_matrix[i, j] = graph.covariance(reward_matrix[i], reward_matrix[j])

cov_df = pd.DataFrame(cov_matrix, index=labels, columns=labels)
print("Kovariansmatrix for branch lengths:")
print(cov_df.round(4))

In [ ]:
# Korrelationsmatrix
stds = np.sqrt(np.diag(cov_matrix))
corr_matrix = cov_matrix / np.outer(stds, stds)
corr_matrix = np.nan_to_num(corr_matrix)  # NaN for 4'ton (std=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(cov_df.round(3), annot=True, fmt=".3f", cmap='coolwarm',
            center=0, ax=axes[0])
axes[0].set_title("Kovariansmatrix")

corr_df = pd.DataFrame(corr_matrix, index=labels, columns=labels)
sns.heatmap(corr_df.round(3), annot=True, fmt=".3f", cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Korrelationsmatrix")
plt.tight_layout()
plt.show()

## Del 5 – PDF for gren længder-fordelinger

Via reward-transformation kan jeg beregne den fulde fordeling (PDF/CDF) for hver SFS-komponent ikke blot forventningsværdien. Dette er særlig nyttigt for inferens. 

### Spørgsmål 5

- Er PDF'en for singleton grenlængde forskellig fra en simpel eksponentiel? Hvad siger formen om den underliggende genealogi?

Hypotese: 

- Fordi singleton grenlængde er en reward-transformation over flere tilstande, forventer jeg en blanding af eksponentielle fordelinger ikke en ren eksponentiel.

In [ ]:
times = np.arange(0, 5, 0.05)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, (rewards, lab) in enumerate(zip(reward_matrix[:-1], labels[:-1])):
    rt_graph = graph.reward_transform(rewards)
    axes[0].plot(times, rt_graph.pdf(times), label=lab)
    axes[1].plot(times, rt_graph.cdf(times), label=lab)

axes[0].set_title("PDF for branch length-typer")
axes[0].set_xlabel("Tid")
axes[0].set_ylabel("Tæthed")
axes[0].legend()
axes[1].set_title("CDF for branch length-typer")
axes[1].set_xlabel("Tid")
axes[1].legend()
plt.tight_layout()
plt.show()

## Del 6 – Populationsstørrelse $N$ og skalering

I den skalerede koalescent absorberes populationsstørrelsen $N$ i tidsskalaen. Den reelle koalescensrate med $k$ lineager er $\binom{k}{2}/N$, og jeg kan parametrisere modellen med $N$ som en fri parameter.

### Spørgsmål 6

- Skalerer $\mathbb{E}[H]$ lineært med $N$, og skalerer $\text{Var}[H]$ med $N^2$?

Hypotese: 

- Da alle rater skaleres med $1/N$, forventer jeg $\mathbb{E}[H] = N \cdot \mathbb{E}[H]_{N=1}$ og $\text{Var}[H] = N^2 \cdot \text{Var}[H]_{N=1}$.

In [ ]:
# Parameteriseret model: koalescensrate = C(k,2) / N
@with_ipv([nr_samples] + [0] * (nr_samples - 1))
def coalescent_param(state):
    transitions = []
    for i in range(state.size):
        for j in range(i, state.size):
            same = int(i == j)
            if same and state[i] < 2: continue
            if not same and (state[i] < 1 or state[j] < 1): continue
            new = state.copy()
            new[i] -= 1; new[j] -= 1; new[i+j+1] += 1
            # Coefficient vector: [C(k,2), 0] — rate multipliceres med 1/N
            transitions.append((new, [state[i] * (state[j] - same) / (1 + same), 0]))
    return transitions

g_param = Graph(coalescent_param)

# Basisværdier ved N=1
g_param.update_weights([1.0, 0])
E_base  = g_param.expectation()
Var_base = g_param.variance()

N_values = [0.5, 1, 2, 5, 10, 50]
rows = []
for N in N_values:
    g_param.update_weights([1/N, 0])
    E   = g_param.expectation()
    Var = g_param.variance()
    rows.append({'N': N,
                 'E[H] numerisk': round(E, 4),
                 'N * E_base': round(N * E_base, 4),
                 'Var[H] numerisk': round(Var, 4),
                 'N² * Var_base': round(N**2 * Var_base, 4)})

pd.DataFrame(rows).set_index('N')

In [ ]:
# Visualiser skalering
N_range = np.linspace(0.1, 20, 200)
E_vals, Var_vals = [], []
for N in N_range:
    g_param.update_weights([1/N, 0])
    E_vals.append(g_param.expectation())
    Var_vals.append(g_param.variance())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(N_range, E_vals, label='Numerisk')
axes[0].plot(N_range, N_range * E_base, '--', label=f'N × E_base ({E_base:.2f})')
axes[0].set_xlabel('N'); axes[0].set_ylabel('E[H]')
axes[0].set_title('Lineær skalering af E[H] med N')
axes[0].legend()

axes[1].plot(N_range, Var_vals, label='Numerisk')
axes[1].plot(N_range, N_range**2 * Var_base, '--', label=f'N² × Var_base ({Var_base:.2f})')
axes[1].set_xlabel('N'); axes[1].set_ylabel('Var[H]')
axes[1].set_title('Kvadratisk skalering af Var[H] med N')
axes[1].legend()
plt.tight_layout()
plt.show()

## Del 7 – Diskrete mutationer og Wattersons estimator

Via *discretize* kan vi tilføje diskrete mutationer med rate $\mu$ langs koalescenttræets grene. Dette giver mig den diskrete fase-type fordeling for det samlede antal adskilte sites $S$.

### Spørgsmål 7

- Er $\mathbb{E}[S] = \mathbb{E}[L] \cdot \mu = a_n \cdot \theta$, dvs. svarer den numeriske forventning til Wattersons estimator?

Hypotese: 

 - Under uendelige-sites modellen er $\mathbb{E}[S] = \theta \cdot a_n$, hvor $\theta = 2N\mu$ og $a_n = \sum_{k=1}^{n-1} 1/k$. Vi undersøger, om $\mathbb{E}[S]$ fra den diskrete phase-type model stemmer overens med dette (Hobolth et al., 2024, Afsnit 4).

In [ ]:
from typing import Optional

def mutation_rate_fn(state: np.ndarray, mutation_rate: float):
    """Mutationsrate proportional med antal linjer (infinite-sites model)"""
    return sum(state) * mutation_rate

mutation_graph = Graph(coalescent)
mu = 0.1
mutation_graph = mutation_graph.discretize(mutation_rate_fn, mutation_rate=mu)
rewards = mutation_graph.rewards

E_S_num = mutation_graph.expectation(rewards)
a_n = sum(1/k for k in range(1, nr_samples))
E_S_th  = E_L_num * mu  # E[L] * mu

print(f"mu = {mu}")
print(f"E[S] numerisk          = {E_S_num:.6f}")
print(f"E[L] * mu analytisk    = {E_S_th:.6f}")
print(f"theta * a_n (theta=2N*mu, N=1) = {2*mu * a_n:.6f}")

In [ ]:
# PMF for antal adskilte sites under forskellige (N, mu)
g_mut = Graph(coalescent_param)

def mu_fn(state, mu=None):
    return sum(state) * mu

g_mut = g_mut.discretize(mu_fn, mu=0.1)  # plads holder

N_mu_combos = [(1, 0.05), (1, 0.2), (2, 0.05), (2, 0.2), (5, 0.05), (5, 0.2)]
x = np.arange(15)

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (N, mu) in zip(axes.flat, N_mu_combos):
    g_mut.update_weights([1/N, mu])
    rt = g_mut.reward_transform(g_mut.rewards)
    pmf = rt.pdf(x)
    ax.bar(x, pmf, color='steelblue', alpha=0.8)
    ax.set_title(f'N={N}, μ={mu}')
    ax.set_xlabel('Antal mutationer S')
    ax.set_ylabel('P(S = s)')
    E_S = g_mut.expectation(g_mut.rewards)
    ax.axvline(E_S, color='red', linestyle='--', label=f'E[S]={E_S:.2f}')
    ax.legend(fontsize=8)
plt.suptitle('PMF for antal adskilte sites under varierende N og μ', fontsize=13)
plt.tight_layout()
plt.show()

## Del 8 – Linje-dynamik over tid

Med *state_probability(t)* kan jeg beregne sandsynligheden for at befinde sig i hver tilstand til tid $t$, og dermed visualisere den forventede linje-dynamik over tid.

### Spørgsmål 8

- Hvordan ser fordelingen af linjer over tid ud, og hvornår er det mest sandsynligt at have præcis $k$ linjer tilbage?

In [ ]:
graph = Graph(coalescent)
times = np.arange(0, 5, 0.04)
state_probs = np.array([graph.state_probability(t) * graph.states().T for t in times])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].stackplot(times, np.sum(state_probs, axis=2).T,
                  labels=labels, alpha=0.8)
axes[0].set_xlabel('Tid'); axes[0].set_ylabel('Forventet antal lineager')
axes[0].set_title('Stackplot: lineage-sammensætning over tid')
axes[0].legend(loc='upper right')

for i, lab in enumerate(labels):
    axes[1].plot(times, np.sum(state_probs[:, i, :], axis=1), label=lab)
axes[1].set_xlabel('Tid'); axes[1].set_ylabel('Forventet antal')
axes[1].set_title('Forventet antal per lineage-type over tid')
axes[1].legend()
plt.tight_layout()
plt.show()